In [260]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import figurefirst as fifi

plt.rcParams['font.serif'] = ['Times'] + plt.rcParams['font.serif']
plt.rcParams['text.usetex'] = False
plt.rcParams["ps.usedistiller"] = 'xpdf'
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.weight'] = 'normal'
plt.rcParams["mathtext.fontset"] = 'cm'


In [261]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os
from typing import Optional
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from braid_analysis import braid_analysis_plots


from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis import unifying_algo_plots as uap

import numpy as np
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial.distance import cdist

from shapely.geometry import MultiPoint, Point
from shapely.ops import unary_union
import numpy as np
from sklearn.preprocessing import MinMaxScaler

from shapely.geometry import MultiPoint, Point
from shapely.ops import unary_union

def format_ticks_as_latex(ax, axis='both'):
    """
    Update tick labels on a matplotlib axis to be wrapped in dollar signs
    for LaTeX rendering.
    
    Parameters:
        ax:     A matplotlib Axes object.
        axis:   Which axis to format: 'x', 'y', or 'both' (default).
    """
    def to_latex(label_text):
        text = label_text.strip()
        if text and not (text.startswith('$') and text.endswith('$')):
            return f'${text}$'
        return text

    def apply_latex_labels(mpl_axis):
        # Draw the figure to ensure tick labels are populated
        ax.figure.canvas.draw()
        labels = [tick.get_text() for tick in mpl_axis.get_ticklabels()]
        new_labels = [to_latex(l) for l in labels]
        mpl_axis.set_ticklabels(new_labels)

    if axis in ('x', 'both'):
        apply_latex_labels(ax.xaxis)
    if axis in ('y', 'both'):
        apply_latex_labels(ax.yaxis)

def to_sentence_case(text: str) -> str:
    if not text:
        return text
    return text[0].upper() + text[1:].lower()

def find_file(directory: str, str1: str, str2: str) -> Optional[str]:
    for filename in os.listdir(directory):
        if str1 in filename and str2 in filename:
            return os.path.join(directory, filename)
    return None

def plot_arrowhead_trajectory_scaled(x, y, color='black', arrow_length=0.05, arrow_angle=30,
                                     ax=None, linewidth=1, scale_bar=False, units='',
                                     flow_direction=None, fontsize=5,
                                     flow_arrow_length=0.05, flow_arrow_angle=30,
                                     flow_arrow_size=0.08, padding=0.2,
                                     flow_column_width=0.2,
                                     flow_arrow_label_gap=2):

    has_flow = flow_direction is not None and flow_arrow_length is not None

    # Get the physical size of the axis in inches to compute aspect-equal coordinate ranges
    ax.figure.canvas.draw()
    bbox = ax.get_window_extent().transformed(ax.figure.dpi_scale_trans.inverted())
    ax_width_in  = bbox.width   # physical width in inches
    ax_height_in = bbox.height  # physical height in inches

    # With set_aspect('equal'), the coordinate range ratio must match the physical ratio.
    # We choose the coordinate space to be [0, ax_width_in] x [0, ax_height_in]
    # (i.e. 1 unit = 1 inch), which naturally satisfies aspect='equal'.
    coord_width  = ax_width_in
    coord_height = ax_height_in

    # Reserve right column for flow arrow
    flow_col = flow_column_width * coord_width if has_flow else 0.0

    # # Left region for trajectory and scale bar (in coordinate space)
    # left_x_min = padding * coord_width
    # left_x_max = coord_width - flow_col - padding * coord_width
    # y_min_pad  = padding * coord_height
    # y_max_pad  = coord_height - padding * coord_height

    # # Scale trajectory into the left region
    # x_norm = (x - np.nanmin(x)) / (np.nanmax(x) - np.nanmin(x))
    # y_norm = (y - np.nanmin(y)) / (np.nanmax(y) - np.nanmin(y))
    # x_scaled = x_norm * (left_x_max - left_x_min) + left_x_min
    # y_scaled = y_norm * (y_max_pad  - y_min_pad)  + y_min_pad

    # # Set axis limits explicitly to our coordinate space before plotting
    # ax.set_xlim(0, coord_width)
    # ax.set_ylim(0, coord_height)

    # Left region for trajectory and scale bar (in coordinate space)
    left_x_min = padding * coord_width
    left_x_max = coord_width - flow_col - padding * coord_width
    y_min_pad  = padding * coord_height
    y_max_pad  = coord_height - padding * coord_height
    
    # Normalize both axes by the same factor to preserve aspect ratio
    x_range = np.nanmax(x) - np.nanmin(x)
    y_range = np.nanmax(y) - np.nanmin(y)
    
    available_w = left_x_max - left_x_min
    available_h = y_max_pad - y_min_pad
    
    # Single scale factor: the most constrained axis wins
    scale = min(available_w / x_range, available_h / y_range)
    
    x_norm = (x - np.nanmin(x)) * scale
    y_norm = (y - np.nanmin(y)) * scale
    
    # Center the scaled data within the available region
    x_offset = left_x_min + (available_w - x_range * scale) / 2
    y_offset = y_min_pad  + (available_h - y_range * scale) / 2
    
    x_scaled = x_norm + x_offset
    y_scaled = y_norm + y_offset
    
    # Set axis limits explicitly to our coordinate space before plotting
    ax.set_xlim(0, coord_width)
    ax.set_ylim(0, coord_height)
    
    ax.set_aspect('equal')

    braid_analysis_plots.plot_arrowhead_trajectory(x_scaled, y_scaled, color=color, arrow_length=arrow_length,
                               arrow_angle=arrow_angle, ax=ax, linewidth=linewidth)

    for collection in ax.collections:
        collection.set_clip_on(False)

    # Reread limits in case set_aspect nudged them
    ax.figure.canvas.draw()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    x_extent_ax = xlim[1] - xlim[0]
    y_extent_ax = ylim[1] - ylim[0]

    label_gap = 0.03 * y_extent_ax
    offset_x  = 0.02 * x_extent_ax

    # Right column bounds
    right_x_min = coord_width - flow_col
    right_x_max = coord_width

    ax.set_aspect('equal')

    # Occupied geometry: convex hull of trajectory
    traj_hull = MultiPoint(list(zip(x_scaled, y_scaled))).convex_hull.buffer(offset_x * 2)
    occupied_geom = traj_hull

    def find_best_position(size, occupied, x_min, x_max, y_min, y_max):
        n = 30
        xs = np.linspace(x_min + size, x_max - size, n)
        ys = np.linspace(y_min + size, y_max - size, n)
        best_pt, best_dist = None, -1
        for cx_ in xs:
            for cy_ in ys:
                pt = Point(cx_, cy_)
                d = pt.distance(occupied) if not occupied.is_empty else 1e9
                box_fits = (cx_ - size > x_min and cx_ + size < x_max and
                            cy_ - size > y_min and cy_ + size < y_max)
                if box_fits and d > best_dist:
                    best_dist = d
                    best_pt = (cx_, cy_)
        return best_pt

    # --- Scale bar ---
    if scale_bar:
        x_extent_original = np.nanmax(x) - np.nanmin(x)
        max_bar_original = 0.3 * x_extent_original
        magnitude = 10 ** np.floor(np.log10(max_bar_original))
        nice_steps = [1, 2, 5]
        bar_size_original = magnitude
        for step in nice_steps:
            candidate = step * magnitude
            if candidate <= max_bar_original:
                bar_size_original = candidate

        scale_factor = (left_x_max - left_x_min) / x_extent_original
        bar_size_scaled = bar_size_original * scale_factor
        bar_half = bar_size_scaled / 2

        pos = find_best_position(max(bar_half, label_gap * 2), occupied_geom,
                                 xlim[0], right_x_min, ylim[0], ylim[1])
        if pos is not None:
            cx, cy = pos
            bar_x_start = cx - bar_half
            bar_x_end   = cx + bar_half

            if cy < (ylim[0] + ylim[1]) / 2:
                text_y, va = cy + label_gap, 'bottom'
            else:
                text_y, va = cy - label_gap, 'top'

            ax.plot([bar_x_start, bar_x_end], [cy, cy], color=color, linewidth=0.5)
            ax.text(cx, text_y, f'{bar_size_original:g} {units}',
                    ha='center', va=va, fontsize=fontsize, color=color)

            bar_geom = MultiPoint([
                (bar_x_start, cy), (bar_x_end, cy), (cx, text_y)
            ]).convex_hull.buffer(label_gap * 2)
            occupied_geom = unary_union([occupied_geom, bar_geom])

    # --- Flow direction arrow ---
    if has_flow:
        arrow_size_scaled = flow_arrow_size * x_extent_ax

        cx = (right_x_min + right_x_max) / 2
        cy = (ylim[0] + ylim[1]) / 2

        dx = np.cos(flow_direction) * arrow_size_scaled / 2
        dy = np.sin(flow_direction) * arrow_size_scaled / 2

        n_points = 10
        t = np.linspace(-0.5, 0.5, n_points)
        arrow_x = cx + t * dx * 2
        arrow_y = cy + t * dy * 2

        braid_analysis_plots.plot_arrowhead_trajectory(
            arrow_x, arrow_y,
            color=color,
            arrow_length=flow_arrow_length,
            arrow_angle=flow_arrow_angle,
            ax=ax,
            linewidth=linewidth
        )

        for collection in ax.collections:
            collection.set_clip_on(False)

        text_angle_deg = np.degrees(flow_direction)
        if 90 < text_angle_deg % 360 < 270:
            text_angle_deg += 180

        
        perp_dx = -np.sin(flow_direction) * label_gap * flow_arrow_label_gap
        perp_dy =  np.cos(flow_direction) * label_gap * flow_arrow_label_gap
        cx_mid = (right_x_min + right_x_max) / 2
        cy_mid = (ylim[0] + ylim[1]) / 2
        if (cx + perp_dx - cx_mid)**2 + (cy + perp_dy - cy_mid)**2 < \
           (cx - perp_dx - cx_mid)**2 + (cy - perp_dy - cy_mid)**2:
            perp_dx, perp_dy = -perp_dx, -perp_dy

        ax.text(cx + perp_dx, cy + perp_dy, 'flow',
                ha='center', va='center', fontsize=fontsize, color=color,
                rotation=text_angle_deg, rotation_mode='anchor')



In [262]:
def get_flow_color(flow_condition: str) -> str:
    colors = {
        'laminar':   '#b83f3fff',
        'turbulent': '#bfd6e8ff',
        'unsteady': '#bfd6e8ff',
        'stillair':  '#084a72ff',
        'still':     '#084a72ff',
        'unknown_circle':   '#8a7d89be',
        'unknown_cast':  '#201c20b8',
    }
    return colors.get(flow_condition, '#000000ff')  # defaults to black if not found

In [263]:
AXIS_RATIO_THRESHOLD = 0.25
FIGURE_NAME = 'animal_summary.svg'

In [264]:
def get_trajectory_filenames(animal_df):
    species = animal_df.species.values[0]

    if species == 'albatross':
        if animal_df.flow_condition.values[0] == 'unknown':
            if animal_df.axis_ratio.values[0] < AXIS_RATIO_THRESHOLD:
                animal_df.flow_condition.values[0] = 'unknown_cast'
            else:
                animal_df.flow_condition.values[0] = 'unknown_circle'
    
    
    objid = animal_df.objid.values[0]
    fname = find_file('animal_data/'+species, objid, 'trajec.parquet')
    print(fname)
    trajec = pd.read_parquet(fname) 
    flow_condition = animal_df.flow_condition.values[0]
    print(flow_condition)
    color = get_flow_color(flow_condition)
    return species, trajec, color, flow_condition, animal_df

In [265]:
df = pd.read_parquet('df_all_species.parquet')
objids_to_drop = ['shark_b', 'shark_c', 'mouse_tortuous', ]
df = df[~df['objid'].isin(objids_to_drop)]

In [266]:
df

,axis_ratio,slope,species,objid,body_length,group,flow_speed,movement_speed,reynolds,flow_condition,distance_travelled,visual_acuity_cpd
0,0.272127,0.130774,eels,eel,0.500,aquatic,0.21,0.108465,159232.281746,laminar,4.447047,3.00
1,0.384097,2.611401,mosquitoes,mosquito_trajec,0.005,aerial,0.40,0.234114,211.371279,laminar,2.402008,0.20
2,0.123890,0.049217,albatross,albatross_casting_trajec,0.900,aerial,NaN,12.982051,NaN,unknown,6361.204976,30.00
3,0.726327,0.125089,albatross,albatross_circling_trajec,0.900,aerial,NaN,7.088350,NaN,unknown,2126.505008,30.00
4,0.231655,0.337006,nautilus,nautilus_yellow,0.120,aquatic,0.07,0.114232,22107.786820,laminar,3.998104,0.06
5,0.595384,3.862881,sharks,shark_a,0.500,aquatic,0.00,0.362324,181162.083513,still,2.485544,5.00
8,0.172558,2.690567,sharks,shark_d,0.500,aquatic,0.15,0.372091,261045.592677,laminar,4.673465,5.00
9,0.699893,5.200235,RL_agents,rl_agent_c,NaN,aerial,0.50,2.858896,NaN,laminar,18.925888,NaN
10,0.173304,2.232564,mouse,mouse_straight,0.070,terrestrial,NaN,0.307772,NaN,trail,4.856646,0.55
12,0.567955,0.741575,terns,terns_Tern1838-1-1,0.300,aerial,NaN,6.086464,NaN,unknown,241.023981,20.00


In [267]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [268]:
for objid in df.objid.values:
    df_animal = df[df.objid==objid]
    species, df_trajec, color, flow_condition, animal_df = get_trajectory_filenames(df_animal)

    if species not in ['albatross', 'nautilus']:
        species_nos = animal_df.species.values[0].rstrip('s')
    else:
        species_nos = species

    try:
        fifi_label = species_nos + '_' + flow_condition
    except:
        fifi_label = species_nos + '_' + objid
        
    if fifi_label not in layout.axes.keys():
        print('no label: ' + fifi_label)
    
    print('  ')

    ax = layout.axes[fifi_label]

    x = df_trajec[df_trajec.time_relative_to_flash>0].x.values
    y = df_trajec[df_trajec.time_relative_to_flash>0].y.values

    flow_direction = df_trajec.flow_direction.values[0]

    if flow_condition not in ['laminar', 'unknown_cast', ]:
        flow_direction = None

    # flip rightward arrows and trajectories
    if flow_direction == 0:
        flow_direction = np.pi
        x *= -1
        y *= -1
    
    plot_arrowhead_trajectory_scaled(x, y, color=color, arrow_length=0.07, arrow_angle=30,
                                         ax=ax, linewidth=0.75, scale_bar=True, units='m', 
                                     flow_direction=flow_direction,
                                     flow_arrow_length=0.05, flow_arrow_angle=45,
                                    flow_arrow_size=0.15, fontsize=4, padding=0.1,
                                 flow_column_width=0.2)

    ax.set_aspect('equal')
    
    fifi.mpl_functions.adjust_spines(ax, [])

animal_data/eels/eel_trajec.parquet
laminar
  
animal_data/mosquitoes/mosquito_trajec_trajec.parquet
laminar
  
animal_data/albatross/albatross_casting_trajec_trajec.parquet
unknown_cast
  
animal_data/albatross/albatross_circling_trajec_trajec.parquet
unknown_circle
  
animal_data/nautilus/nautilus_yellow_trajec.parquet
laminar
  
animal_data/sharks/shark_a_trajec.parquet
still
  
animal_data/sharks/shark_d_trajec.parquet
laminar
  
animal_data/RL_agents/rl_agent_c_trajec.parquet
laminar
  
animal_data/mouse/mouse_straight_trajec.parquet
trail
  
animal_data/terns/terns_Tern1838-1-1_trajec.parquet
unknown
  
animal_data/crabs/crab_12710_trajec.parquet
laminar
  
animal_data/moths/1Oct08E_trajec.parquet
turbulent
  
animal_data/moths/1Oct08B_trajec.parquet
laminar
  
animal_data/nautical_search/charlie_sierra_trajec.parquet
None
  
animal_data/nautical_search/sierra_sierra_trajec.parquet
None
  
animal_data/d_melanogaster/d_mel_laminar_20220927_170437_3462_123_trajec.parquet
laminar
  

In [269]:
layout.append_figure_to_layer(layout.figures['none'], 'trajectory_plots', cleartarget=True)

In [270]:
layout.write_svg(FIGURE_NAME)

# Aligned Course plots

In [ ]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [ ]:
directories = {'eels': '', 
               'mosquitoes': '', 
               'albatross': ['casting', 'circling'], 
               'nautilus': '', 
               'sharks': ['_a_', '_b_', '_c_', '_d_'], 
               'RL_agents': '',
               'mouse': ['straight', 'tort'],
               'terns': '',
               'crabs': ['12710'],
               'moths': ['1Oct08E', '1Oct08B'], 
               'nautical_search': ['charlie_sierra', 'sierra_sierra'],
               'd_melanogaster': ['d_mel_laminar_20220927_170437_3462_123',
                                  'd_mel_stillair_20220815_162329_5322_124',
                                  'd_mel_unsteady_20240110_164820_11858_151',
                                 ],
               'd_sechelia': '',
               'walking_drosophila': ['walking_fly_0_4',],
               'cockroach': '',
              }


In [ ]:
import os
from pathlib import Path
from typing import Tuple, Optional

# Alternative version that returns full paths instead of just filenames
def find_parquet_files_fullpath(directory: str, key: Optional[str] = None) -> Tuple[Optional[str], Optional[str]]:
    """
    Find files ending with 'trajec.parquet' and 'unifying_analysis.parquet' in a directory.
    
    Args:
        directory: Path to the directory to search
        key: Optional string that must be present in the filename (not path) to match
        
    Returns:
        A tuple containing full paths (trajec_file, unifying_analysis_file)
        Returns None for any file not found
    """
    trajec_file = None
    unifying_file = None

    directory = os.path.join('animal_data', directory)
    directory_path = Path(directory)
    
    if not directory_path.is_dir():
        raise ValueError(f"Directory not found: {directory}")
    
    for file_path in directory_path.iterdir():
        if file_path.is_file():
            # Skip if key is specified and not in filename
            if key is not None and key not in file_path.name:
                continue
                
            if file_path.name.endswith('trajec.parquet'):
                trajec_file = str(file_path)
            elif file_path.name.endswith('unifying_analysis.parquet'):
                unifying_file = str(file_path)
            
            if trajec_file and unifying_file:
                break
    
    return trajec_file, unifying_file
    
def load_trajec_and_unifying(directory, keyword):
    trajec_file, unifying_file = find_parquet_files_fullpath(directory, keyword)
    trajec_df = pd.read_parquet(trajec_file)
    unifying_df = pd.read_parquet(unifying_file)
    return trajec_df, unifying_df

In [ ]:
if 0:
    fig_casting = plt.figure(figsize=(4,4))
    ax_casting = fig_casting.add_subplot(111)
    
    fig_circling = plt.figure(figsize=(4,4))
    ax_circling = fig_circling.add_subplot(111)
else:
    ax_casting = layout.axes[('course', 'casting')]
    ax_circling = layout.axes[('course', 'circling')]

for directory, keywords in directories.items():
    if type(keywords) is str:
        keywords = [keywords,]
    for keyword in keywords:
        trajec_df, unifying_df = load_trajec_and_unifying(directory, keyword)


        # get weighted mean rotation angle
        unifying_df['rotation'] = uaa.wrap_angle(unifying_df['rotation'])
        unifying_df['rotation_positive'] = unifying_df['rotation'].copy()
        ix = np.where(unifying_df['rotation_positive'].values<0)[0]
        unifying_df.loc[ix, 'rotation_positive'] = unifying_df.loc[ix, 'rotation_positive']
        unifying_df['rotation_positive_sine'] = np.sin(unifying_df['rotation_positive'])
        unifying_df['rotation_positive_cosine'] = np.cos(unifying_df['rotation_positive'])
        sin_rot = uaa.get_weighted_value(unifying_df, 'rotation_positive_sine', 'rmse_affine')
        cos_rot = uaa.get_weighted_value(unifying_df, 'rotation_positive_cosine', 'rmse_affine')
        mean_rot = np.arctan2(sin_rot, cos_rot)
        print(mean_rot)
        
        sorted_fit = unifying_df.sort_values('rmse_affine')
        best_ix = sorted_fit.index[0]
        best_unifying_algo_fit = unifying_df.iloc[best_ix]
    
        if np.abs(best_unifying_algo_fit.axis_ratio) < AXIS_RATIO_THRESHOLD:
            ax = ax_casting
        else:
            ax = ax_circling
    
        # slope = rad/frames
        # slope in rad/sec = slope / dt
        # period = 1/slope_in_rad_sec*2pi
        length_of_1_period = 2*np.pi/(best_unifying_algo_fit.slope/best_unifying_algo_fit.timestep_sec)
        length_of_1_period = np.abs(length_of_1_period)
        
        uap.plot_aligned_course(unifying_df,
                                trajec_df,
                                apply_alignment_shift=True,
                                normalize_rotation_to_center=np.pi/2 - mean_rot, #mean_rot - np.pi/2,
                                show_linear_fit=False,
                                show_affine_fit=False,
                                show_flash=False,
                                show_roi=True,
                                flash_frame_start=20,
                                flash_frame_end=88,
                                xlim_start=-4,
                                xlim_end=4,
                                ax=ax,
                                timewarp=best_unifying_algo_fit.timestep_sec/length_of_1_period,
                                clean_spines=False,
                                course_markersize=1,
                                alpha_override=0.2,
                                course_marker_alpha=0.5,
                               )
        ax.set_rasterization_zorder(0)

for i, ax in enumerate([ax_casting, ax_circling]):

    xlim_start = -2
    xlim_end = 2
    ax.set_xlim(xlim_start, xlim_end)
    ax.set_ylim(-np.pi, np.pi)
    
    
    
    ax.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_yticklabels(['$-\pi$', '$-\pi/2$','$0$','$\pi/2$','$\pi$',])
    ax.set_ylabel('Course direction')

    if i == 0:
        fifi.mpl_functions.adjust_spines(ax, ['left'])
    else:
        fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'])
        ax.set_xlabel('Aligned time (relative)')

    fifi.mpl_functions.set_fontsize(ax, 6)

layout.append_figure_to_layer(layout.figures['course'], 'course', cleartarget=True)
layout.write_svg(FIGURE_NAME)